In [ ]:
# === PARAMETERS (injected by papermill) ===

# Experiment Configuration
# "0-1"  = New Decentralized (Self on Apple = 1, Else 0)
# "TEAM" = Report 2 Reward (Self on Apple = -1, Other on Apple = 2/(n-1), Else 0)
REWARD_TYPE = "0-1"

# Split Strategy
# "RANDOM"   = Mixed positions (Hypothesis: Success for MLP)
# "POSITION" = Train on top 80% positions, Test on bottom 20% (Hypothesis: Failure for MLP)
SPLIT_STRATEGY = "RANDOM"

SEED = 42

# Data Generation
WIDTH = 9
HEIGHT = 9
NUM_AGENTS = 4
SAMPLES_PER_POSITION = 100  # Set to 5000 for full run. 100 for quick sanity check.
TRAIN_SPLIT = 0.8           # Fraction of data to train on

# Model Architecture
HIDDEN_DIM = 512
NUM_LAYERS = 4

# Training
LEARNING_RATE = 0.001
BATCH_SIZE = 1024      # Can increase if dataset is large
NUM_ITERATIONS = 100  # Set to 20000+ for full run

# Outputs
OUTPUT_FILE = "mlp_robust_results.csv"

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import collections
import pandas as pd
import os

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
def generate_robust_dataset(width, height, num_agents, samples_per_pos):
    """
    Generates data sequentially by position (0,0 -> 0,1 -> ... -> 8,8).
    Strictly enforces UNIQUENESS of agent configurations per position.
    Enforces 3-way class balance (Picker / Bystander / Zero).
    
    Returns:
        inputs: (N, 3*H*W)
        rewards_01: (N,)
        rewards_team: (N,)
    """
    input_list = []
    r01_list = []
    rTeam_list = []
    
    total_positions = width * height
    print(f"Generating data for {total_positions} positions x {samples_per_pos} samples...")
    
    # Loop sequentially (Important for POSITION split strategy)
    for r in range(height):
        for c in range(width):
            self_pos = (r, c)
            seen_configs = set()
            
            # Target Counts: 1/3 Picker, 1/3 Bystander, 1/3 Zero
            n_picker = samples_per_pos // 3
            n_bystander = samples_per_pos // 3
            n_zero = samples_per_pos - (n_picker + n_bystander)
            
            targets = [("picker", n_picker), ("bystander", n_bystander), ("zero", n_zero)]
            
            for mode, count in targets:
                generated = 0
                while generated < count:
                    # --- 1. Init Grids ---
                    apples_grid = np.zeros((height, width), dtype=np.float32)
                    agents_grid = np.zeros((height, width), dtype=np.float32) # All agents
                    
                    # --- 2. Place Self ---
                    agents_grid[self_pos] = 1.0
                    
                    # --- 3. Place Apple & Define Reward ---
                    if mode == "picker":
                        apple_pos = self_pos
                        apples_grid[apple_pos] = 1.0
                        
                        r01 = 1.0
                        rTeam = -1.0
                        
                    elif mode == "bystander":
                        # Apple must be somewhere else
                        while True:
                            ar, ac = np.random.randint(0, height), np.random.randint(0, width)
                            if (ar, ac) != self_pos:
                                apple_pos = (ar, ac)
                                break
                        apples_grid[apple_pos] = 1.0
                        
                        # Someone ELSE must be on the apple
                        agents_grid[apple_pos] += 1.0 
                        
                        r01 = 0.0
                        rTeam = 2.0 / (num_agents - 1)
                        
                    else: # mode == "zero"
                        # Apple somewhere else
                        while True:
                            ar, ac = np.random.randint(0, height), np.random.randint(0, width)
                            if (ar, ac) != self_pos:
                                apple_pos = (ar, ac)
                                break
                        apples_grid[apple_pos] = 1.0
                        
                        r01 = 0.0
                        rTeam = 0.0

                    # --- 4. Place Remaining Agents (Noise) ---
                    # We placed Self. If Bystander, we placed 1 Other.
                    agents_placed = 1 + (1 if mode == "bystander" else 0)
                    agents_needed = num_agents - agents_placed
                    
                    other_positions_for_hash = []
                    if mode == "bystander":
                        other_positions_for_hash.append(apple_pos)
                        
                    for _ in range(agents_needed):
                        pr, pc = np.random.randint(0, height), np.random.randint(0, width)
                        agents_grid[pr, pc] += 1.0
                        other_positions_for_hash.append((pr, pc))
                        
                    # --- 5. Uniqueness Check ---
                    # Hash (ApplePos, FrozenSet(OtherAgents))
                    # Self is fixed for this outer loop, so we don't need it in hash
                    config_hash = hash((apple_pos, tuple(sorted(other_positions_for_hash))))
                    
                    if config_hash not in seen_configs:
                        seen_configs.add(config_hash)
                        
                        # Create Input: [Apples, Others, Self] flattened
                        self_grid = np.zeros((height, width), dtype=np.float32)
                        self_grid[self_pos] = 1.0
                        others_grid = agents_grid - self_grid
                        
                        inp = np.concatenate([apples_grid.flatten(), others_grid.flatten(), self_grid.flatten()])
                        
                        input_list.append(inp)
                        r01_list.append(r01)
                        rTeam_list.append(rTeam)
                        
                        generated += 1
                        
    return np.array(input_list), np.array(r01_list), np.array(rTeam_list)

# Generate Data
X_all, Y01_all, YTeam_all = generate_robust_dataset(WIDTH, HEIGHT, NUM_AGENTS, SAMPLES_PER_POSITION)
print(f"Dataset Shape: {X_all.shape}")

In [ ]:
# 1. Select Target Reward
if REWARD_TYPE == "0-1":
    Y_target = Y01_all
    print("Task: New Decentralized Reward (0-1)")
else:
    Y_target = YTeam_all
    print("Task: Team Reward (-1, 2/n-1, 0)")

X_tensor = torch.tensor(X_all, dtype=torch.float32)
Y_tensor = torch.tensor(Y_target, dtype=torch.float32)

total_samples = len(X_tensor)
split_idx = int(total_samples * TRAIN_SPLIT)

# 2. Apply Split Strategy
if SPLIT_STRATEGY == "RANDOM":
    print("Strategy: RANDOM split (Full Coverage). Expecting Success.")
    # Global Shuffle
    perm = torch.randperm(total_samples)
    train_indices = perm[:split_idx]
    test_indices = perm[split_idx:]
    
elif SPLIT_STRATEGY == "POSITION":
    print("Strategy: POSITION split (Train on top rows, Test on bottom). Expecting Failure.")
    # NO SHUFFLE initially. Data is generated by position order.
    # We take the first X% chunks (positions) as train.
    all_indices = torch.arange(total_samples)
    train_indices = all_indices[:split_idx]
    test_indices = all_indices[split_idx:]
    
    # We must shuffle the TRAIN set internally so batches aren't homogeneous
    train_perm = torch.randperm(len(train_indices))
    train_indices = train_indices[train_perm]

# Create Sets
X_train, Y_train = X_tensor[train_indices], Y_tensor[train_indices]
X_test, Y_test = X_tensor[test_indices], Y_tensor[test_indices]

print(f"Train samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")

In [22]:
# === CELL 4.5: DATA LEAKAGE VERIFICATION ===
def verify_integrity(train_t, test_t):
    print("Verifying data integrity...")
    # Convert to numpy and hash rows
    train_hashes = set([bytes(row.cpu().numpy()) for row in train_t])
    test_hashes = set([bytes(row.cpu().numpy()) for row in test_t])
    
    # Intersection check
    overlap = train_hashes.intersection(test_hashes)
    print(f"Train Unique: {len(train_hashes)}")
    print(f"Test Unique:  {len(test_hashes)}")
    print(f"Overlap:      {len(overlap)}")
    
    if len(overlap) > 0:
        raise ValueError(f"CRITICAL: Found {len(overlap)} overlapping states!")
    print("✅ SUCCESS: Train and Test sets are strictly disjoint.")

verify_integrity(X_train, X_test)

Verifying data integrity...
Train Unique: 6480
Test Unique:  1620
Overlap:      0
✅ SUCCESS: Train and Test sets are strictly disjoint.


In [ ]:
class RewardMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

input_dim = 3 * WIDTH * HEIGHT
model = RewardMLP(input_dim, HIDDEN_DIM, NUM_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

print(model)

In [ ]:
losses = []
model.train()

print(f"Starting Training ({NUM_ITERATIONS} steps)...")

# Helper for batching
def get_batch(x, y, batch_size):
    idx = torch.randint(0, len(x), (batch_size,))
    return x[idx].to(device), y[idx].to(device)

pbar = tqdm(range(NUM_ITERATIONS))
for i in pbar:
    bx, by = get_batch(X_train, Y_train, BATCH_SIZE)
    
    optimizer.zero_grad()
    preds = model(bx).squeeze(1)
    loss = loss_fn(preds, by)
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if i % 100 == 0:
        pbar.set_description(f"Loss: {loss.item():.5f}")

plt.figure(figsize=(10,4))
plt.plot(losses)
plt.title("Training Loss")
plt.yscale('log')
plt.ylabel("MSE")
plt.xlabel("Iteration")
plt.show()

In [ ]:
def evaluate_metrics(name, x, y):
    model.eval()
    with torch.no_grad():
        # Process in chunks to prevent OOM
        chunk_size = 5000
        preds_list = []
        for i in range(0, len(x), chunk_size):
            bx = x[i:i+chunk_size].to(device)
            out = model(bx).squeeze(1)
            preds_list.append(out.cpu())
        
        preds = torch.cat(preds_list)
        y = y.cpu()
        
    # --- Metrics ---
    # 1. Non-Zero Targets (Picker/Bystander) -> MAPE
    mask_nonzero = (y.abs() > 1e-6) # float tolerance
    if mask_nonzero.sum() > 0:
        denom = y[mask_nonzero].abs()
        diff = (preds[mask_nonzero] - y[mask_nonzero]).abs()
        mape = (diff / denom).mean().item() * 100
    else:
        mape = 0.0
        
    # 2. Zero Targets -> MAE
    mask_zero = (y.abs() <= 1e-6)
    if mask_zero.sum() > 0:
        mae_zero = (preds[mask_zero] - y[mask_zero]).abs().mean().item()
    else:
        mae_zero = 0.0
        
    print(f"[{name}] NonZero N={mask_nonzero.sum().item()} | MAPE: {mape:.4f}%")
    print(f"[{name}] Zero    N={mask_zero.sum().item()} | MAE:  {mae_zero:.6f}")
    
    return mape, mae_zero

print("--- Final Evaluation ---")
train_mape, train_mae = evaluate_metrics("TRAIN", X_train, Y_train)
test_mape, test_mae = evaluate_metrics("TEST", X_test, Y_test)

# Save Results
results = {
    "reward_type": REWARD_TYPE,
    "split_strategy": SPLIT_STRATEGY,
    "samples_per_pos": SAMPLES_PER_POSITION,
    "iterations": NUM_ITERATIONS,
    "train_mape": train_mape,
    "train_mae_zero": train_mae,
    "test_mape": test_mape,
    "test_mae_zero": test_mae,
    "train_samples": len(X_train),
    "test_samples": len(X_test)
}

df = pd.DataFrame([results])
header = not os.path.exists(OUTPUT_FILE)
df.to_csv(OUTPUT_FILE, mode='a', header=header, index=False)
print(f"Results appended to {OUTPUT_FILE}")